In [1]:
from manim import *

video_scene = " -v WARNING --disable_caching srt_Scene"
image_scene = f" -v WARNING --disable_caching -r {2*427},{2*240}  -s srt_Scene"

In [158]:
class Clock(Mobject):
	def __init__(self, clock_center = np.array([0, 0, 0]), clock_radius = 1, **kwargs):
		super().__init__(**kwargs)

		self.clock_center = clock_center
		self.clock_radius = clock_radius

		clock_circle_filled = Circle(radius = self.clock_radius, color = WHITE, fill_color = WHITE, fill_opacity = 0.9).move_to(self.clock_center)
		clock_circle = Circle(radius = self.clock_radius, color = BLACK).move_to(self.clock_center)
		self.add(clock_circle_filled, clock_circle)

		clock_tick_line_group = VGroup()
		for i in range(12):
			clock_tick_line = Line(start = clock_center+clock_radius*UP, end = clock_center+clock_radius*UP*0.8, color = BLACK).rotate(angle = -i * 2*PI / 12, about_point = clock_center)
			clock_tick_line_group.add(clock_tick_line)
		self.add(clock_tick_line_group)


	def get_time(self, time = 0):
		time_group = VGroup()

		time_pointer = Line(
			start = self.clock_center, end = self.clock_center+self.clock_radius*0.7*UP, color = RED, stroke_width = 6
			).add_tip(tip_width = 0.0655, tip_length = 0.1).rotate(angle = -time * 2*PI / 60 * 10, about_point = self.clock_center)
		time_group.add(time_pointer)


		time_center_circle = Circle(radius = 0.025, color = GRAY, fill_color = GRAY).move_to(self.clock_center)
		time_group.add(time_center_circle)
		return time_group



class Light(Mobject):
	def __init__(self, light_start = np.array([0, 0, 0]), direction = RIGHT, light_color = RED, **kwargs):
		super().__init__(**kwargs)

		self.light_start = light_start
		self.direction = direction
		self.light_color = light_color

		self.lamp = ImageMobject("../external_media/lamp.png").scale(0.2).move_to(light_start)
		self.lamp.shift((self.lamp.get_right()-self.lamp.get_center()-0.5)*LEFT)
		self.lamp_left = ImageMobject("../external_media/lamp_left.png").scale(0.2).move_to(light_start)
		self.lamp_left.shift((self.lamp_left.get_center()-self.lamp_left.get_left()-0.5)*RIGHT)
		if (self.direction[0] < 0):
			self.lamp = self.lamp_left
		self.add(self.lamp)


	def get_ray(self, length = 1):
		# sine to the right
		if (self.direction[0] > 0):
			sine_graph = FunctionGraph(lambda x: 0.1*np.sin(20*x), x_range = [0, length], color = self.light_color, stroke_width = 3)
			sine_graph.move_to(self.lamp.get_right()+self.direction/2*RIGHT)
		# sine to the left
		else:
			sine_graph = FunctionGraph(lambda x: -0.1*np.sin(20*x), x_range = [-length, 0], color = self.light_color, stroke_width = 3)
			sine_graph.move_to(self.lamp.get_left()+self.direction/2*RIGHT)

		return sine_graph




class SpaceShip(Mobject):
	def __init__(self, ship_center = np.array([0, 0, 0]), ship_width = 5, ship_height = 2, **kwargs):
		super().__init__(**kwargs)

		self.ship_center = ship_center
		self.ship_width = ship_width
		self.ship_height = ship_height

		# spaceship
		self.spaceship_group = Group()
		spaceship_rect = Rectangle(width = ship_width, height = ship_height, stroke_color = BLACK).move_to(ship_center)
		self.spaceship_group.add(spaceship_rect)

		# parameters
		self.clock_height = 0.75
		self.light_height = 1.65

		# left clock
		self.left_clock_center = spaceship_rect.get_corner(DOWN+LEFT)+self.clock_height*UP
		self.left_clock = Clock(clock_center = self.left_clock_center, clock_radius = 0.5)
		self.left_time = self.left_clock.get_time(0)
		self.spaceship_group.add(self.left_clock)

		# right clock
		self.right_clock_center = spaceship_rect.get_corner(DOWN+RIGHT)+self.clock_height*UP
		self.right_clock = Clock(clock_center = self.right_clock_center, clock_radius = 0.5)
		self.right_time = self.right_clock.get_time(0)
		self.spaceship_group.add(self.right_clock)

		# left lamp
		self.light_left = Light(light_start = spaceship_rect.get_corner(DOWN)+self.light_height*UP, direction = LEFT)
		self.spaceship_group.add(self.light_left)

		# right lamp
		self.light_right = Light(light_start = spaceship_rect.get_corner(DOWN)+self.light_height*UP)
		self.spaceship_group.add(self.light_right)
		self.add(self.spaceship_group, self.left_time, self.right_time)


	def set_right_time(self, time):
		# self.remove(self.right_time)
		self.right_time.become(self.right_clock.get_time(time))
	

	def set_left_time(self, time):
		self.left_time.become(self.left_clock.get_time(time))


	def get_left_lamp(self, time):
		return

	def get_right_lamp(self, time):
		return
		


	def move_ship(self, velocity = 1, time = 0):
		self.left_time.shift(RIGHT * velocity * time)
		self.right_time.shift(RIGHT * velocity * time)
		self.spaceship_group.move_to(self.ship_center + RIGHT * velocity * time)


class Earth(Mobject):
	def __init__(self, earth_center = np.array([0, 0, 0]), **kwargs):
		super().__init__(**kwargs)

		self.earth_center = earth_center
		self.earth = ImageMobject("../external_media/earth.png").scale(0.5).move_to(earth_center)
		self.add(self.earth)


	def move_earth(self, velocity = 1, time = 0):
		self.earth.move_to(self.earth_center + RIGHT * velocity * time)


In [159]:
%%manim -qm --fps 60 $video_scene


class srt_Scene(Scene):
	def construct(self):
		self.camera.background_color = WHITE

		CVC = Text('CVC', font_size = 12, weight = BOLD, color = WHITE, font = 'Latin Modern Sans').align_on_border(RIGHT + DOWN, buff = 0.2)
		self.add(CVC)

		# test clock
		# clock = Clock(clock_center = np.array([-3, -2, 0]), clock_radius = 0.5)
		# clock_time = clock.get_time(15)
		# self.add(clock, clock_time)

		spaceship = SpaceShip(ship_center = np.array([-3, 1.5, 0]))
		self.add(spaceship)

		# light_right = Light()
		# light_left = Light(light_start = np.array([0, 0, 0]), direction = LEFT)
		# self.add(light_right, light_left)

		# light_ray_right = light_right.get_ray()
		# light_ray_left = light_left.get_ray()
		# self.add(light_ray_right, light_ray_left)

		earth = Earth(earth_center = np.array([4, -3.5, 0]))
		self.add(earth)


		def spaceship_updater(ship):
			t = time_tracker.get_value()
			ship.set_right_time(t)
			ship.set_left_time(t)
			ship.move_ship(time = t)


		time_tracker = ValueTracker(0)
		self.wait(1.5)

		spaceship.add_updater(spaceship_updater)
		self.play(time_tracker.animate.set_value(6), rate_func = linear, run_time = 5)
		self.wait(3)

Manim Community v0.18.1